In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import time
import numpy as np

In [3]:
import sys
import os

# Go up one level to the project root so Python can find 'moogp' and 'ogp'
sys.path.append(os.path.abspath('..'))

from moogp.model import MOOGP

In [4]:
import numpy as np

def test_eigenvalue_clipping():
    # 1. Setup a highly correlated scenario
    n, p, q = 200, 5, 2
    rng = np.random.default_rng(42)
    
    # Points very close together (highly collinear)
    X = np.sort(rng.uniform(0, 0.1, size=(n, 1)), axis=0) 
    Y = rng.normal(size=(n, p))
    
    # 2. Extract SVD (Dk will be naturally large-ish)
    _, svals, Vt = np.linalg.svd(Y, full_matrices=False)
    Phi = Vt.T[:, :q] * np.sqrt(n) / svals[:q]
    d_vals = n / (svals[:q]**2)
    
    # 3. Build a spatial matrix with an EXTREME lengthscale
    # This simulates the optimizer testing a bad hyperparameter
    dist_sq = (X - X.T)**2
    ell_extreme = 100.0  # Huge lengthscale
    C_star = np.exp(-dist_sq / (2 * ell_extreme**2)) # RBF Kernel for simplicity

    # 4. The Math WITHOUT Clipping
    Wk_unclipped, _ = np.linalg.eigh(C_star)
    
    # 5. The Math WITH Clipping
    Wk_clipped = np.maximum(Wk_unclipped, 1e-10)

    print(f"Smallest unclipped eigenvalue: {np.min(Wk_unclipped)}")
    print(f"Smallest clipped eigenvalue:   {np.min(Wk_clipped)}\n")

    # Let's see what happens to the log-determinant formula:
    Dk = d_vals[0]
    print(f"Data Multiplier (Dk): {Dk:.2f}")

    try:
        # np.log will throw a runtime warning if it hits a negative number
        with np.errstate(invalid='raise'):
            logdet_unclipped = np.sum(np.log(1.0 + Dk * Wk_unclipped))
            print(f"Unclipped LogDet: {logdet_unclipped}")
    except FloatingPointError:
        print("UNCLIPPED FAILED: Encountered np.log(negative number) -> NaN")

    try:
        logdet_clipped = np.sum(np.log(1.0 + Dk * Wk_clipped))
        print(f"CLIPPED SUCCESS:  LogDet = {logdet_clipped:.4f}")
    except FloatingPointError:
        print("Clipped Failed.")

test_eigenvalue_clipping()

Smallest unclipped eigenvalue: -6.736740927222133e-14
Smallest clipped eigenvalue:   1e-10

Data Multiplier (Dk): 0.77
Unclipped LogDet: 5.048340480322718
CLIPPED SUCCESS:  LogDet = 5.0483
